# KWISMO — Collecte de données (scraping + OCR)

Ce notebook n'écrit aucune logique lui-même : il appelle uniquement le code de `src/data/` (scrape.py, scrape_social.py, ocr.py, metrics.py). Le projet exige **Python 3.13** (voir `src/__init__.py`).

Ce notebook s'adapte automatiquement à votre environnement (**Local**, **Google Colab** ou **Kaggle Notebooks**).

In [ ]:
# Détection automatique de l'environnement (Local, Google Colab, Kaggle Notebooks)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB:
    ENV_NAME = "Google Colab"
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive
            print("⚡ Connexion automatique à Google Drive...")
            drive.mount('/content/drive')
        except Exception as err:
            print(f"⚠️ Montage Google Drive recommandé : {err}")
elif ON_KAGGLE:
    ENV_NAME = "Kaggle Notebooks"
else:
    ENV_NAME = "Local"

print(f"Environnement de calcul détecté : {ENV_NAME}")

In [ ]:
# Gestion du répertoire de travail et clônage du dépôt sur les plateformes Cloud
if ON_COLAB:
    PROJECT_DIR = Path("/content/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /content/kwismo
    else:
        !git -C /content/kwismo fetch && git -C /content/kwismo reset --hard origin/main
    os.chdir(PROJECT_DIR)
elif ON_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /kaggle/working/kwismo
    else:
        !git -C /kaggle/working/kwismo fetch && git -C /kaggle/working/kwismo reset --hard origin/main
    os.chdir(PROJECT_DIR)
else:
    PROJECT_DIR = Path.cwd()
    if (PROJECT_DIR / "kwismo-ai").exists():
        PROJECT_DIR = PROJECT_DIR / "kwismo-ai"
        os.chdir(PROJECT_DIR)

print("Dossier de travail :", PROJECT_DIR)

In [ ]:
# Installation de Python 3.13 et création du venv isolé (.venv313)
VENV_DIR = PROJECT_DIR / ".venv313"

if ON_COLAB or ON_KAGGLE:
    python313_bin = VENV_DIR / "bin" / "python"
    if not python313_bin.exists():
        print("Installation de Python 3.13 et création de l'environnement .venv313...")
        !apt-get update -y
        !apt-get install -y software-properties-common
        !add-apt-repository -y ppa:deadsnakes/ppa
        !apt-get update -y
        !apt-get install -y python3.13 python3.13-venv python3.13-dev
        !python3.13 -m venv {VENV_DIR}
        !{VENV_DIR}/bin/pip install --upgrade pip
        !{VENV_DIR}/bin/pip install -r requirements.txt
        !{VENV_DIR}/bin/python -m playwright install --with-deps chromium
    PYTHON_BIN = str(python313_bin)
else:
    PYTHON_BIN = sys.executable

def run_module(module: str) -> None:
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a échoué (code {result.returncode})")
    else:
        print(result.stdout)

ver_proc = subprocess.run([PYTHON_BIN, "--version"], capture_output=True, text=True)
print("Interprète Python configuré :", PYTHON_BIN)
print("Version vérifiée :", ver_proc.stdout.strip() or ver_proc.stderr.strip())

## 1. Scraping web dynamique

Découverte de pages via mots-clés, extraction du texte principal, téléchargement des images des articles pour OCR, et dédoublonnage atomique (SQLite).

In [ ]:
run_module("src.data.scrape")

## 2. Extraction OCR sur les images scrapées

Extraction du texte des captures et images d'articles via EasyOCR, enregistrement dans `messages.jsonl`.

In [ ]:
run_module("src.data.ocr")

## 3. Métriques et visualisations de collecte

Affichage de l'historique et des graphes (taux d'OCR, volume par domaine, etc.).

In [ ]:
run_module("src.data.metrics")